In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()


Instructions for updating:
non-resource variables are not supported in the long term


# 선형 회귀(Linear Regression)

선형 회귀 분석은 알려진 데이터를 사용해서 알 수 없는 데이터를 `예측`하는 데이터 분석 기법으로 가장 기본적인 머신러닝 방법이며, 여러 분야에서 사용된다.

알 수 없는 변수(종속 변수)와 알려진 변수(독립 변수)를 선형 방정식으로 수학적으로 모델링한다.  
피처와 레이블 간의 선형 관계를 파악하는 알고리즘으로 피처 데이터($x$, 독립 변수)와 피처 데이터에 따른 레이블($y$, 종속 변수) 사이의 선형 관계가 존재할 때 이를 수식화 하면 아래와 같다.

$$y=ax+b=wx+b$$

피처 데이터($x$)가 주어질 때, $x$에 가중치(weight) $w$를 곱하고 바이어스($b$)를 더하면 피처 데이터에 따른 레이블($y$)을 얻을 수 있다. 선형 회귀에서 해야할 일은 $x$와 $y$를 이용해서 최적의 가중치 $w$와 바이어스 $b$를 구하는 것이다.

위의 수식은 피처의 개수가 1개일 경우이고, 이를 일반화시켜 $p$개의 피처를 가지는 데이터라고 가정하면 데이터셋을 구성하는 각 데이터는 $p$개의 학습 데이터로 구성되므로 $x=(x_1, x_2, x_3, ..., x_p)$라고 표현할 수 있다. $x_i$는 i번째 데이터를 열 벡터로 표현한 것이다. 그래서 선형 회귀 모델은 아래와 같다.

$$y=w_1x_1+w_2x_2+w_3x_3+...+w_px_p+b$$

## 학습 데이터를 만든다.

In [2]:
xData = [1, 2, 3, 4, 5, 6, 7, 8] # 근무 시간, 피처
yData = [25000, 55000, 75000, 110000, 128000, 155000, 180000, 232500] # 근무 시간에 따른 매출 금액, 레이블(실제값)

## 과적합 방지하기 위해 난수를 발생시켜 가중치와 바이어스의 초기값을 설정한다.

$y=wx+b$를 만족시키는 $w$(기울기, 가중치)와 $b$(y절편, 바이어스)를 과적합을 방지하기 위해 난수를 발생시켜 정해야 한다. 과적합이란 학습한 데이터내에서는 높은 정확도를 보이지만 학습한 데이터 이외의 데이터에서서는 낮은 정확도를 보이는 문제점을 말한다.

random_uniform() 메소드로 최대값과 최소값 사이에 지정한 개수의 난수를 발생시킨다.  
random_uniform([난수의 차원], 최소값, 최대값)

In [3]:
w = tf.Variable(tf.random_uniform([1], -100, 100)) # 가중치, -100 ~ 100 사이의 난수
b = tf.Variable(tf.random_uniform([1], -100, 100)) # 바이어스, -100 ~ 100 사이의 난수
print(f'w = {w}, b = {b}')

w = <tf.Variable 'Variable:0' shape=(1,) dtype=float32_ref>, b = <tf.Variable 'Variable_1:0' shape=(1,) dtype=float32_ref>


tensorflow 변수를 초기화하고 난수로 발생시킨 가중치와 바이어를 확인한다.

In [4]:
sess = tf.Session()
sess.run(tf.global_variables_initializer())
print(f'w = {sess.run(w)}, b = {sess.run(b)}')

w = [-37.69319], b = [-3.9245834]


## 피처(근무 시간)와 피처에 따른 레이블(매출 금액)을 저장할 placeholder를 선언한다.

In [5]:
x = tf.placeholder(dtype=tf.float32) # 피처(xData)를 저장할 placeholder를 선언한다.
y = tf.placeholder(dtype=tf.float32) # 피처에 따른 레이블(yData)을 저장할 placeholder를 선언한다.

## 가설($y=wx+b$)을 세운다.

In [6]:
Y = w * x + b

## 최적의 가중치와 바이어스를 찾기 위한 오차 함수(손실, 비용)를 정의한다.

실제값(yData => y)과 예측값(Y)의 편차에 대한 제곱의 평균을 이용해서 평균 제곱 오차(Mean Squared Error, MSE)를 계산하는 오차 함수를 정의한다.  
square() 메소드로 tensorflow에서 제곱값을 계산한다.  
reduce_mean() 메소드로 tensorflow에서 평균을 계산한다.  
sqrt() 메소드로 tensorflow에서 제곱근을 계산한다.

In [7]:
loss = tf.reduce_mean(tf.square(Y - y))

## 경사 하강법 알고리즘을 이요해서 오차 함수의 결과값을 가장 작게만드는 방향을 학습하도록 정의한다.

In [8]:
# 경사 하강법 알고리즘의 학습율(learning rate)을 설정한다.
lr = tf.Variable(0.001)
# train.GradientDescentOptimizer 클래스로 학습율을 넘겨서 경사 하강법 알고리즘 객체를 만든다.
# 경사 하강법 알고리즘 객체에서 minimize() 메소드로 오차 함수를 넘겨서 오차 함수의 결과값을 가장 작게만드는 최적의 가중치와 바이어스를 찾는다.
train = tf.train.GradientDescentOptimizer(lr).minimize(loss)

## 학습시킨다.

In [9]:
# 세션을 만들고 변수를 초기화시킨다.
sess = tf.Session()
sess.run(tf.global_variables_initializer())

# 일정 횟수(epoch)만큼 학습시켜서 최적의 가중치와 바이어스를 찾는다.
for i in range(20001):
    # 오차 함수의 결과를 가장 작게만드는 학습을 할 수 있도록 placeholder에 데이터를 넣어주고 학습시킨다.
    sess.run(train, feed_dict={x: xData, y: yData})
    # 일정 횟수마다 중간 결과를 출력한다.
    if i % 2000 == 0:
        print('epoch: {:5d}, loss: {:14.2f}, w: {:9.2f}, b: {:9.2f}'.format(i, sess.run(loss, feed_dict={x: xData, y: yData}), sess.run(w)[0], sess.run(b)[0]))

epoch:     0, loss: 16696715264.00, w:   1324.88, b:    261.83
epoch:  2000, loss:    60511764.00, w:  27025.88, b:   -681.18
epoch:  4000, loss:    57371864.00, w:  27453.65, b:  -3086.19
epoch:  6000, loss:    56736176.00, w:  27646.10, b:  -4168.28
epoch:  8000, loss:    56607480.00, w:  27732.71, b:  -4655.15
epoch: 10000, loss:    56581424.00, w:  27771.68, b:  -4874.24
epoch: 12000, loss:    56576128.00, w:  27789.20, b:  -4972.81
epoch: 14000, loss:    56575060.00, w:  27797.10, b:  -5017.22
epoch: 16000, loss:    56574804.00, w:  27800.67, b:  -5037.23
epoch: 18000, loss:    56574776.00, w:  27802.24, b:  -5046.15
epoch: 20000, loss:    56574784.00, w:  27802.94, b:  -5050.10


In [10]:
print('9시간 근무했을 때 예상 매출 금액: {:,.0f}원'.format(sess.run(w)[0] * 9 + sess.run(b)[0]))

9시간 근무했을 때 예상 매출 금액: 245,176원


In [11]:
for i in range(9, 25):
    print('{:2d}시간 근무했을 때 예상 매출 금액: {:,.0f}원'.format(i, sess.run(Y, feed_dict={x: [i]})[0]))

 9시간 근무했을 때 예상 매출 금액: 245,176원
10시간 근무했을 때 예상 매출 금액: 272,979원
11시간 근무했을 때 예상 매출 금액: 300,782원
12시간 근무했을 때 예상 매출 금액: 328,585원
13시간 근무했을 때 예상 매출 금액: 356,388원
14시간 근무했을 때 예상 매출 금액: 384,191원
15시간 근무했을 때 예상 매출 금액: 411,994원
16시간 근무했을 때 예상 매출 금액: 439,797원
17시간 근무했을 때 예상 매출 금액: 467,600원
18시간 근무했을 때 예상 매출 금액: 495,403원
19시간 근무했을 때 예상 매출 금액: 523,206원
20시간 근무했을 때 예상 매출 금액: 551,009원
21시간 근무했을 때 예상 매출 금액: 578,812원
22시간 근무했을 때 예상 매출 금액: 606,615원
23시간 근무했을 때 예상 매출 금액: 634,418원
24시간 근무했을 때 예상 매출 금액: 662,220원
